In [1]:
print("hello")

hello


# Installing Dependencies

In [2]:
pip install transformers peft accelerate datasets

Note: you may need to restart the kernel to use updated packages.


In [3]:
# dependency issue -- `torch_dtype` is deprecated! Use `dtype` instead!
!pip install bitsandbytes>=0.46.1

In [4]:
!pip install -U trl==0.12.0 --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.2/310.2 kB 17.6 MB/s eta 0:00:00


# DATASET ALTERATION FOR SFT NOW

In [1]:
import json
from pathlib import Path
from collections import defaultdict
import random


INPUT_PATH = "/kaggle/input/datasets/ibtihussain/pref-pair-sft-dpo/preference_pairs.jsonl"
TRAIN_OUTPUT_PATH = "/kaggle/working/sft_train.jsonl"
HOLDOUT_OUTPUT_PATH = "/kaggle/working/sft_holdout.jsonl"  

SEED = 10
HOLDOUT_FRACTION = 0.2 # 20% MEANS 6 OUT OF 30 TASKS IDs WILL BE FOR EVALS


all_pairs = []
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        all_pairs.append(json.loads(line))

print(f"Loaded {len(all_pairs)} total preference pairs")

Loaded 112 total preference pairs


In [2]:
# --- Split by task_id (not by row) to avoid leakage across train/eval ---
unique_task_ids = sorted(set(p["task_id"] for p in all_pairs))
print(f"Unique task_ids: {len(unique_task_ids)}")

random.seed(SEED)
shuffled_task_ids = unique_task_ids.copy()
random.shuffle(shuffled_task_ids)

num_holdout = max(1, round(len(shuffled_task_ids) * HOLDOUT_FRACTION))
holdout_task_ids = set(shuffled_task_ids[:num_holdout])
train_task_ids = set(shuffled_task_ids[num_holdout:])

print(f"Holdout task_ids ({len(holdout_task_ids)}): {sorted(holdout_task_ids)}")
print(f"Train task_ids ({len(train_task_ids)}): {sorted(train_task_ids)}")

Unique task_ids: 30
Holdout task_ids (6): ['task_05', 'task_08', 'task_10', 'task_18', 'task_21', 'task_23']
Train task_ids (24): ['task_01', 'task_02', 'task_03', 'task_04', 'task_06', 'task_07', 'task_09', 'task_11', 'task_12', 'task_13', 'task_14', 'task_15', 'task_16', 'task_17', 'task_19', 'task_20', 'task_22', 'task_24', 'task_25', 'task_26', 'task_27', 'task_28', 'task_29', 'task_30']


In [3]:
# Build SFT examples: prompt + chosen only (drop rejected for this stage)
def build_sft_record(pair):
    return {
        "task_id": pair["task_id"],
        "tool": pair["tool"],
        "tier": pair["tier"],
        "messages": [
            {"role": "user", "content": pair["prompt"]},
            {"role": "assistant", "content": pair["chosen"]},
        ],
    }

train_records = [build_sft_record(p) for p in all_pairs if p["task_id"] in train_task_ids]
holdout_records = [build_sft_record(p) for p in all_pairs if p["task_id"] in holdout_task_ids]

print(f"\nSFT train examples: {len(train_records)}")
print(f"SFT holdout examples: {len(holdout_records)}")



SFT train examples: 92
SFT holdout examples: 20


In [4]:
with open(TRAIN_OUTPUT_PATH, "w", encoding="utf-8") as f:
    for rec in train_records:
        f.write(json.dumps(rec) + "\n")

with open(HOLDOUT_OUTPUT_PATH, "w", encoding="utf-8") as f:
    for rec in holdout_records:
        f.write(json.dumps(rec) + "\n")

print(f"\nSaved train set to {TRAIN_OUTPUT_PATH}")
print(f"Saved holdout set to {HOLDOUT_OUTPUT_PATH}")


Saved train set to /kaggle/working/sft_train.jsonl
Saved holdout set to /kaggle/working/sft_holdout.jsonl


# LLM for fine tuning

Qwen/Qwen2.5-Coder-3B-Instruct

In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"

# --- 4-bit quantization config (QLoRA) ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

print(model)
print(f"\nModel loaded. Vocab size: {len(tokenizer)}")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear4bit(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm

In [6]:
# Prepare model for k-bit training and attach LoRA adapters
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [7]:
from datasets import Dataset
import json

SFT_TRAIN_PATH = "/kaggle/working/sft_train.jsonl"

# Load the JSONL into a list of dicts 
train_data = []
with open(SFT_TRAIN_PATH, "r", encoding="utf-8") as f:
    for line in f:
        train_data.append(json.loads(line))

print(f"Loaded {len(train_data)} SFT training examples")


# THIS WAS WITHOUT SYSTEM PROMPT TRAINING
# # Qwen's chat template to format each example as a single training string 
# def format_example(example):
#     formatted_text = tokenizer.apply_chat_template(
#         example["messages"],
#         tokenize=False,
#         add_generation_prompt=True,  # ← HOLY MOLYYYY THIS CAUSING THE ERROR IN INFERENCE AND TRAINING CHAT TEMPLATE CHANGED: Now matches inference format
#     )
#     return {"text": formatted_text}


# THIS WAS WITH SYSTEM PROMPT TRAINING
# Qwen's chat template to format each example as a single training string 
def format_example(example):
    messages = [
        {"role": "system", "content": "You are a code reviewer. Follow this exact structure: 1) Briefly validate what works (1-2 sentences), 2) Use 'However' to transition to specific failures, 3) Reference specific test cases (Test 3, Test 5, etc.), 4) Provide concrete fixes. Do NOT provide generic code descriptions or explanations."},
        *example["messages"]
    ]
    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,  # ← HOLY MOLYYYY THIS CAUSING THE ERROR IN INFERENCE AND TRAINING CHAT TEMPLATE CHANGED: Now matches inference format
    )
    return {"text": formatted_text}

formatted_data = [format_example(ex) for ex in train_data]

# token length distribution before committing to a max_seq_length
lengths = [len(tokenizer.encode(ex["text"])) for ex in formatted_data]
print(f"\nToken length stats:")
print(f"  min: {min(lengths)}")
print(f"  max: {max(lengths)}")
print(f"  mean: {sum(lengths)/len(lengths):.0f}")
print(f"  95th percentile: {sorted(lengths)[int(len(lengths)*0.95)]}")

# Build the HF Dataset
train_dataset = Dataset.from_list(formatted_data)
print(f"\nDataset ready: {train_dataset}")
print(f"\nExample formatted text:\n{'-'*50}\n{train_dataset[0]['text'][:800]}")

Loaded 92 SFT training examples

Token length stats:
  min: 202
  max: 627
  mean: 334
  95th percentile: 571

Dataset ready: Dataset({
    features: ['text'],
    num_rows: 92
})

Example formatted text:
--------------------------------------------------
<|im_start|>system
You are a code reviewer. Follow this exact structure: 1) Briefly validate what works (1-2 sentences), 2) Use 'However' to transition to specific failures, 3) Reference specific test cases (Test 3, Test 5, etc.), 4) Provide concrete fixes. Do NOT provide generic code descriptions or explanations.<|im_end|>
<|im_start|>user
Review this code:

def sum_numeric_strings(numbers):
    return sum(float(n) for n in numbers)<|im_end|>
<|im_start|>assistant
The function correctly uses a generator expression with `sum()` and `float()` for a clean, Pythonic approach to numeric conversion. However, it lacks defensive programming for two critical edge cases: (1) the function crashes when `numbers` is `None` instead of handling it 

In [8]:
from trl import SFTTrainer, SFTConfig

MAX_SEQ_LENGTH = 800  # comfortably covers your 568 max with headroom

sft_config = SFTConfig(
    output_dir="/kaggle/working/aicr_sft_checkpoint",
    max_seq_length=MAX_SEQ_LENGTH,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,   # effective batch size = 8
    num_train_epochs=4,  #checking on 4 epochs, 3 epochs was insufficient for training
    learning_rate=2e-4,
    optim="paged_adamw_8bit",
    logging_steps=5,
    save_strategy="epoch",
    bf16=True,
    report_to="none",
    dataset_text_field="text",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

trainer.train()

Map:   0%|          | 0/92 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and br

Step,Training Loss
5,1.759310
10,1.099784
15,0.732024
20,0.582324
25,0.507339
30,0.460841
35,0.430868
40,0.366239
45,0.356981


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

TrainOutput(global_step=48, training_loss=0.6798001602292061, metrics={'train_runtime': 1494.2497, 'train_samples_per_second': 0.246, 'train_steps_per_second': 0.032, 'total_flos': 2403308834045952.0, 'train_loss': 0.6798001602292061, 'epoch': 4.0})

In [9]:
SFT_ADAPTER_PATH = "/kaggle/working/aicr-sft-adapter-v2"

trainer.model.save_pretrained(SFT_ADAPTER_PATH)
tokenizer.save_pretrained(SFT_ADAPTER_PATH)

print(f"SFT adapter saved to {SFT_ADAPTER_PATH}")

SFT adapter saved to /kaggle/working/aicr-sft-adapter-v2


# Fined tuned Qwen's inference sanity check 

In [14]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch, json

MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"
SFT_ADAPTER_PATH = "/kaggle/working/aicr-sft-adapter-v2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# --- Fresh, clean base model
fresh_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

sft_model = PeftModel.from_pretrained(fresh_base_model, SFT_ADAPTER_PATH)
sft_model.eval()

sft_tokenizer = AutoTokenizer.from_pretrained(SFT_ADAPTER_PATH)

# --- Load a couple of held-out examples to test on ---
holdout_examples = []
with open("/kaggle/working/sft_holdout.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        holdout_examples.append(json.loads(line))

print(f"Loaded {len(holdout_examples)} held-out examples")



Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loaded 20 held-out examples


In [15]:
# --- Picked 3 examples from different held-out task_ids ---
seen_tasks = set()
sample_examples = []
for ex in holdout_examples:
    if ex["task_id"] not in seen_tasks:
        sample_examples.append(ex)
        seen_tasks.add(ex["task_id"])
    if len(sample_examples) == 3:
        break

# --- Run inference on each sample ---
def generate_review(model, tokenizer, prompt_text, max_new_tokens=400):
    messages = [{"role": "user", "content": prompt_text}]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


for i, ex in enumerate(sample_examples):
    prompt_text = ex["messages"][0]["content"]
    ground_truth_chosen = ex["messages"][1]["content"]

    print(f"\n{'='*70}")
    print(f"HELD-OUT EXAMPLE {i+1} — task_id: {ex['task_id']}, tool: {ex['tool']}, tier: {ex['tier']}")
    print(f"{'='*70}")
    print(f"\n--- PROMPT (snippet) ---\n{prompt_text[:500]}...")
    print(f"\n--- SFT MODEL OUTPUT ---\n{generate_review(sft_model, sft_tokenizer, prompt_text)}")
    print(f"\n--- REFERENCE (chosen, from your dataset) ---\n{ground_truth_chosen}")


HELD-OUT EXAMPLE 1 — task_id: task_05, tool: gpt, tier: 1

--- PROMPT (snippet) ---
Review this code:

from collections.abc import Mapping


def flatten_json(data, parent_key="", sep="."):
    """
    Flatten a nested JSON-like dictionary into a flat dictionary
    using dot-notation keys.

    Example:
        {"a": {"b": 1}, "c": 2}
        -> {"a.b": 1, "c": 2}
    """
    flat = {}

    for key, value in data.items():
        new_key = f"{parent_key}{sep}{key}" if parent_key else str(key)

        if isinstance(value, Mapping):
            flat.update(flatten_json(value, ne...

--- SFT MODEL OUTPUT ---
The function `flatten_json` is well-structured and follows a clear recursive approach to handle nested dictionaries. The use of `Mapping` from the `collections.abc` module provides type safety and is appropriate for this use case. The function correctly handles both string and numeric values and uses a consistent separator (`.`) for the flattened keys, which is a good choice for rea

# FULL HOLD-OUT TEST SET INFERENCE VIA FINE TUNED MODEL SFT-APADTER-V2

In [5]:
import json
import re
import torch
from pathlib import Path
from datetime import datetime
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# --- CONFIG ---
MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"
SFT_ADAPTER_PATH = "/kaggle/input/datasets/ibtihussain/qwen-sft-v2"   # the SYSTEM-PROMPT-TRAINED adapter
HOLDOUT_PATH = "/kaggle/working/sft_holdout.jsonl"
GROUND_TRUTH_PATH = "/kaggle/input/datasets/ibtihussain/ground-truth/ground_truth.json"     # adjust path if stored elsewhere
OUTPUT_TXT_PATH = "/kaggle/working/holdout_inference_sysprompt_trained.txt"

USE_SYSTEM_MESSAGE_AT_INFERENCE = False  # testing the harder, more realistic condition

SYSTEM_MESSAGE = (
    "You are a code reviewer. Follow this exact structure: "
    "1) Briefly validate what works (1-2 sentences), "
    "2) Use 'However' to transition to specific failures, "
    "3) Reference specific test cases (Test 3, Test 5, etc.), "
    "4) Provide concrete fixes. Do NOT provide generic code descriptions or explanations."
)

# --- Load a completely fresh base model (avoid the variable-reuse bug) ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

fresh_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.bfloat16,
)
sft_model = PeftModel.from_pretrained(fresh_base_model, SFT_ADAPTER_PATH)
sft_model.eval()
sft_tokenizer = AutoTokenizer.from_pretrained(SFT_ADAPTER_PATH)

print("SFT model (system-prompt-trained) loaded fresh. Active adapters:", sft_model.active_adapters)

# --- Load holdout set, dedupe by (task_id, tool) since Tier 1 has multiple rejected variants per prompt ---
holdout_raw = []
with open(HOLDOUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        holdout_raw.append(json.loads(line))

seen = set()
holdout_unique = []
for ex in holdout_raw:
    key = (ex["task_id"], ex["tool"])
    if key not in seen:
        holdout_unique.append(ex)
        seen.add(key)

print(f"Loaded {len(holdout_raw)} holdout rows, {len(holdout_unique)} unique (task_id, tool) examples")

# --- Load ground truth for grounding-accuracy comparison ---
with open(GROUND_TRUTH_PATH, "r", encoding="utf-8") as f:
    ground_truth = json.load(f)

# --- Generation function ---
def generate_review(model, tokenizer, prompt_text, use_system_message, max_new_tokens=400):
    messages = []
    if use_system_message:
        messages.append({"role": "system", "content": SYSTEM_MESSAGE})
    messages.append({"role": "user", "content": prompt_text})

    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


# --- Heuristic scoring functions (proxy metrics, not perfect - see caveat below) ---
def score_structure(response_text):
    """Rough proxy for validate-then-critique structure adherence."""
    text_lower = response_text.lower()
    has_transition = bool(re.search(r"\bhowever\b|\bbut\b", text_lower))
    has_fix_language = bool(re.search(
        r"\badd\b|\bconsider\b|\bguard clause\b|\bshould\b|\bfix\b|\bvalidat", text_lower
    ))
    starts_with_validation = not text_lower.strip().startswith(("###", "1.", "1)", "- **"))
    score = sum([has_transition, has_fix_language, starts_with_validation])
    return {
        "has_transition": has_transition,
        "has_fix_language": has_fix_language,
        "avoids_walkthrough_format": starts_with_validation,
        "structure_score_0_to_3": score,
    }


def extract_key_terms(notes_text):
    """Pulls out likely-important keywords from a ground_truth notes string."""
    notes_lower = notes_text.lower()
    candidate_terms = [
        "none", "empty", "negative", "zero", "malformed", "timezone", "utc",
        "version", "segment", "space", "error", "keyword", "future", "date",
        "type", "length", "boundary", "index", "duplicate",
    ]
    found = [t for t in candidate_terms if t in notes_lower]
    return found


def score_grounding(response_text, ground_truth_notes):
    """Rough proxy: does the response mention the same key failure terms as ground truth?"""
    key_terms = extract_key_terms(ground_truth_notes)
    if not key_terms:
        return {"key_terms_expected": [], "key_terms_found": [], "grounding_score": None}
    response_lower = response_text.lower()
    found_terms = [t for t in key_terms if t in response_lower]
    grounding_score = len(found_terms) / len(key_terms)
    return {
        "key_terms_expected": key_terms,
        "key_terms_found": found_terms,
        "grounding_score": round(grounding_score, 2),
    }


# --- Run inference across the full unique holdout set ---
output_lines = []
output_lines.append(f"AICR Holdout Inference Report — SFT Adapter (SYSTEM PROMPT BAKED INTO TRAINING)")
output_lines.append(f"Generated: {datetime.now().isoformat()}")
output_lines.append(f"System message used at inference: {USE_SYSTEM_MESSAGE_AT_INFERENCE}")
output_lines.append(f"Adapter path: {SFT_ADAPTER_PATH}")
output_lines.append(f"Total unique held-out examples: {len(holdout_unique)}")
output_lines.append("=" * 80)

all_structure_scores = []
all_grounding_scores = []

for i, ex in enumerate(holdout_unique):
    task_id = ex["task_id"]
    tool = ex["tool"]
    prompt_text = ex["messages"][0]["content"]
    reference_chosen = ex["messages"][1]["content"]

    gt_notes = ground_truth.get(task_id, {}).get(tool, {}).get("notes", "")
    gt_label = ground_truth.get(task_id, {}).get(tool, {}).get("label", "")

    model_output = generate_review(
        sft_model, sft_tokenizer, prompt_text, USE_SYSTEM_MESSAGE_AT_INFERENCE
    )

    structure_result = score_structure(model_output)
    grounding_result = score_grounding(model_output, gt_notes)

    all_structure_scores.append(structure_result["structure_score_0_to_3"])
    if grounding_result["grounding_score"] is not None:
        all_grounding_scores.append(grounding_result["grounding_score"])

    output_lines.append(f"\n{'='*80}")
    output_lines.append(f"EXAMPLE {i+1} — task_id: {task_id}, tool: {tool}, label: {gt_label}")
    output_lines.append(f"{'='*80}")
    output_lines.append(f"\n--- PROMPT ---\n{prompt_text[:600]}")
    output_lines.append(f"\n--- GROUND TRUTH NOTES ---\n{gt_notes}")
    output_lines.append(f"\n--- MODEL OUTPUT ---\n{model_output}")
    output_lines.append(f"\n--- REFERENCE CHOSEN (from dataset) ---\n{reference_chosen}")
    output_lines.append(f"\n--- HEURISTIC SCORES ---")
    output_lines.append(f"Structure: {structure_result}")
    output_lines.append(f"Grounding: {grounding_result}")

    print(f"[{i+1}/{len(holdout_unique)}] {task_id}/{tool} — "
          f"structure={structure_result['structure_score_0_to_3']}/3, "
          f"grounding={grounding_result['grounding_score']}")

# --- Summary ---
avg_structure = sum(all_structure_scores) / len(all_structure_scores) if all_structure_scores else 0
avg_grounding = sum(all_grounding_scores) / len(all_grounding_scores) if all_grounding_scores else 0

output_lines.append(f"\n{'='*80}")
output_lines.append("SUMMARY")
output_lines.append(f"{'='*80}")
output_lines.append(f"Average structure score (0-3 scale): {avg_structure:.2f}")
output_lines.append(f"Average grounding score (0-1 scale, keyword-overlap proxy): {avg_grounding:.2f}")
output_lines.append(f"Examples scored for grounding: {len(all_grounding_scores)} / {len(holdout_unique)}")

# --- Save to file ---
with open(OUTPUT_TXT_PATH, "w", encoding="utf-8") as f:
    f.write("\n".join(output_lines))

print(f"\nSaved full report to {OUTPUT_TXT_PATH}")
print(f"Average structure score: {avg_structure:.2f}/3")
print(f"Average grounding score: {avg_grounding:.2f}")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


SFT model (system-prompt-trained) loaded fresh. Active adapters: ['default']
Loaded 20 holdout rows, 10 unique (task_id, tool) examples
[1/10] task_05/gpt — structure=2/3, grounding=1.0
[2/10] task_08/gpt — structure=3/3, grounding=0.33
[3/10] task_08/cursor — structure=3/3, grounding=0.33
[4/10] task_23/gpt — structure=3/3, grounding=0.5
[5/10] task_23/cursor — structure=3/3, grounding=0.0
[6/10] task_05/cursor — structure=3/3, grounding=1.0
[7/10] task_10/cursor — structure=3/3, grounding=1.0
[8/10] task_18/gpt — structure=3/3, grounding=0.5
[9/10] task_21/claude — structure=1/3, grounding=1.0
[10/10] task_08/claude — structure=1/3, grounding=None

Saved full report to /kaggle/working/holdout_inference_sysprompt_trained.txt
Average structure score: 2.50/3
Average grounding score: 0.63


# BELOW CELLS ARE FOR SOLEY FOR TESTING CONFIGURATIONS OF LORA, QUANTIZATION, BASE MODEL, SFT ADAPTERS

In [ ]:
# --- Picked 3 examples from different held-out task_ids ---
seen_tasks = set()
sample_examples = []
for ex in holdout_examples:
    if ex["task_id"] not in seen_tasks:
        sample_examples.append(ex)
        seen_tasks.add(ex["task_id"])
    if len(sample_examples) == 3:
        break

# --- Run inference on each sample ---
def generate_review(model, tokenizer, prompt_text, max_new_tokens=400):
    messages = [
        {"role": "system", "content": "You are a code reviewer. Follow this exact structure: 1) Briefly validate what works (1-2 sentences), 2) Use 'However' to transition to specific failures, 3) Reference specific test cases (Test 3, Test 5, etc.), 4) Provide concrete fixes. Do NOT provide generic code descriptions or explanations."},
        {"role": "user", "content": prompt_text}
    ]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


for i, ex in enumerate(sample_examples):
    prompt_text = ex["messages"][0]["content"]
    ground_truth_chosen = ex["messages"][1]["content"]

    print(f"\n{'='*70}")
    print(f"HELD-OUT EXAMPLE {i+1} — task_id: {ex['task_id']}, tool: {ex['tool']}, tier: {ex['tier']}")
    print(f"{'='*70}")
    print(f"\n--- PROMPT (snippet) ---\n{prompt_text[:500]}...")
    print(f"\n--- SFT MODEL OUTPUT ---\n{generate_review(sft_model, sft_tokenizer, prompt_text)}")
    print(f"\n--- REFERENCE (chosen, from your dataset) ---\n{ground_truth_chosen}")

In [10]:
import shutil



shutil.make_archive("/kaggle/working/aicr-sft-adapter-v2", "zip", "/kaggle/working/aicr-sft-adapter-v2")
print("Zipped and ready to download.")

Zipped and ready to download.


SFT QWEN ADAPTERS NOT BEHAIVING WELL ACCORDING TO THE DATASET, NOT TRAINED WELL ENOUGH 


* Test greedy decoding + longer output (free, no retraining) — rule this out first
* Increase epochs (5-8) — free, just retrain longer on the same data
* Increase LoRA rank (16→32) — free, just retrain with a bigger adapter
* Strengthen the instruction in the prompt itself (e.g., explicitly say "review with validation then critique" rather than just "review this code") — free, just changes prompt phrasing
* Only after 1-4 — if the behavior still doesn't shift, then genuinely consider whether you need more/better data (e.g., expanding Tier 1 rejected variants, or adding more distinct task examples)

In [ ]:
print(model)

In [ ]:
from peft import PeftModel
import torch

SFT_ADAPTER_PATH = "/kaggle/working/aicr_sft_checkpoint/checkpoint-48"


sft_model = PeftModel.from_pretrained(model, SFT_ADAPTER_PATH)
sft_model.eval()

sft_tokenizer = AutoTokenizer.from_pretrained(SFT_ADAPTER_PATH)

# --- Load a couple of held-out examples to test on ---
trainset_examples = []
with open("/kaggle/working/sft_train.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        trainset_examples.append(json.loads(line))

print(f"Loaded {len(trainset_examples)} trainset examples")


# --- Picked 3 examples from different held-out task_ids ---
seen_tasks = set()
sample_examples = []
for ex in trainset_examples:
    if ex["task_id"] not in seen_tasks:
        sample_examples.append(ex)
        seen_tasks.add(ex["task_id"])
    if len(sample_examples) == 3:
        break

# --- Run inference on each sample ---
def generate_review(model, tokenizer, prompt_text, max_new_tokens=400):
    messages = [{"role": "user", "content": prompt_text}]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


for i, ex in enumerate(sample_examples):
    prompt_text = ex["messages"][0]["content"]
    ground_truth_chosen = ex["messages"][1]["content"]

    print(f"\n{'='*70}")
    print(f"HELD-OUT EXAMPLE {i+1} — task_id: {ex['task_id']}, tool: {ex['tool']}, tier: {ex['tier']}")
    print(f"{'='*70}")
    print(f"\n--- PROMPT (snippet) ---\n{prompt_text[:500]}...")
    print(f"\n--- SFT MODEL OUTPUT ---\n{generate_review(sft_model, sft_tokenizer, prompt_text)}")
    print(f"\n--- REFERENCE (chosen, from your dataset) ---\n{ground_truth_chosen}")

**Weights diagnostic tests**

In [ ]:
from peft import PeftModel
import torch

# Load checkpoint-24
model_24 = PeftModel.from_pretrained(model, "/kaggle/working/aicr_sft_checkpoint/checkpoint-24")

# Grab one LoRA weight tensor as a fingerprint
weight_24 = None
for name, param in model_24.named_parameters():
    if "lora_A" in name and "q_proj" in name:
        weight_24 = param.clone().detach()
        print(f"Found: {name}, shape: {weight_24.shape}")
        break

print("checkpoint-24 sample values:", weight_24.flatten()[:5])

In [ ]:
model_96 = PeftModel.from_pretrained(model, "/kaggle/working/aicr_sft_checkpoint/checkpoint-96")

weight_96 = None
for name, param in model_96.named_parameters():
    if "lora_A" in name and "q_proj" in name:
        weight_96 = param.clone().detach()
        break

print("checkpoint-96 sample values:", weight_96.flatten()[:5])
print("Are they identical?", torch.equal(weight_24, weight_96))

### Debugging Inference: Explicitly Loading Base Model and SFT Adapter

The previous inference cells were likely not applying the LoRA adapter correctly because `PeftModel.from_pretrained` was called on an already adapted model object. To confirm if the adapter is working, we will explicitly load the base model for inference and then attach the `checkpoint-96` adapter.

We'll compare the output of the raw base model with the SFT fine-tuned model on a held-out example.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import json

MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"
SFT_ADAPTER_PATH = "/kaggle/working/aicr_sft_checkpoint/checkpoint-96"

# --- 4-bit quantization config (QLoRA) - Must be the same as during training ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# --- Reload Base Model and Tokenizer for clean inference ---
inference_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if inference_tokenizer.pad_token is None:
    inference_tokenizer.pad_token = inference_tokenizer.eos_token

inference_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

# --- Load SFT adapter onto the freshly loaded base model ---
inference_sft_model = PeftModel.from_pretrained(inference_base_model, SFT_ADAPTER_PATH)
inference_sft_model.eval() # Set to evaluation mode

print(f"Successfully loaded base model and SFT adapter from {SFT_ADAPTER_PATH}")

In [ ]:
# --- Define the generate_review function ---
def generate_review(model_to_use, tokenizer_to_use, prompt_text, max_new_tokens=400):
    messages = [{"role": "user", "content": prompt_text}]
    input_text = tokenizer_to_use.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer_to_use(input_text, return_tensors="pt").to(model_to_use.device)

    with torch.no_grad():
        output_ids = model_to_use.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=False,
            pad_token_id=tokenizer_to_use.eos_token_id,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer_to_use.decode(generated, skip_special_tokens=True)

# --- Load a held-out example to test on ---
holdout_examples = []
with open("/kaggle/working/sft_holdout.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        holdout_examples.append(json.loads(line))

# Pick one example for direct comparison
sample_example = holdout_examples[0] # Using the first held-out example
prompt_text = sample_example["messages"][0]["content"]
ground_truth_chosen = sample_example["messages"][1]["content"]

print(f"\n{'='*70}")
print(f"COMPARISON: Base Model vs. SFT Model on HELD-OUT EXAMPLE")
print(f"task_id: {sample_example['task_id']}, tool: {sample_example['tool']}, tier: {sample_example['tier']}")
print(f"{'='*70}")

print(f"\n--- PROMPT (snippet) ---\n{prompt_text[:500]}...")

print(f"\n--- BASE MODEL OUTPUT ---\n{generate_review(inference_base_model, inference_tokenizer, prompt_text)}")

print(f"\n--- SFT MODEL (checkpoint-96) OUTPUT ---\n{generate_review(inference_sft_model, inference_tokenizer, prompt_text)}")

print(f"\n--- REFERENCE (chosen, from your dataset) ---\n{ground_truth_chosen}")

Debugging Inference: Merging LoRA Weights into the Base Model
Since the LoRA weights are confirmed to be different, but the model output is not changing, it suggests the adapter might not be effectively applied during generation. To bypass any potential issues with PeftModel's runtime application, we will explicitly merge the LoRA weights into the base model for inference.

This process will create a new model object where the fine-tuned weights are directly incorporated into the base layers, ensuring that generate uses the fine-tuned parameters.

We will compare:

The original base model's output.
The merged SFT model's output.
The reference (chosen) output from your dataset.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import json

MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"
SFT_ADAPTER_PATH = "/kaggle/working/aicr_sft_checkpoint/checkpoint-96"

# --- 4-bit quantization config (QLoRA) - Must be the same as during training ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# --- Reload Base Model and Tokenizer for clean inference ---
merged_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if merged_tokenizer.pad_token is None:
    merged_tokenizer.pad_token = merged_tokenizer.eos_token

# Load the base model without the adapter first for a baseline comparison
original_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
original_base_model.eval()

# Load the base model again, then attach and merge the adapter
model_to_merge = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

# Load SFT adapter onto this new base model
peft_model_for_merge = PeftModel.from_pretrained(model_to_merge, SFT_ADAPTER_PATH)

# Merge the adapter weights into the base model
merged_sft_model = peft_model_for_merge.merge_and_unload()
merged_sft_model.eval() # Set to evaluation mode

print(f"Successfully loaded original base model.")
print(f"Successfully loaded and merged SFT adapter from {SFT_ADAPTER_PATH} into a new model.")

In [ ]:
# --- Define the generate_review function (re-using for clarity) ---
def generate_review_merged(model_to_use, tokenizer_to_use, prompt_text, max_new_tokens=400):
    messages = [{"role": "user", "content": prompt_text}]
    input_text = tokenizer_to_use.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer_to_use(input_text, return_tensors="pt").to(model_to_use.device)

    with torch.no_grad():
        output_ids = model_to_use.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=False,
            pad_token_id=tokenizer_to_use.eos_token_id,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer_to_use.decode(generated, skip_special_tokens=True)

# --- Load a held-out example to test on ---
holdout_examples = []
with open("/kaggle/working/sft_holdout.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        holdout_examples.append(json.loads(line))

# Pick one example for direct comparison (using the same one as before)
sample_example = holdout_examples[0]
prompt_text = sample_example["messages"][0]["content"]
ground_truth_chosen = sample_example["messages"][1]["content"]

print(f"\n{'='*70}")
print(f"COMPARISON: Original Base Model vs. MERGED SFT Model on HELD-OUT EXAMPLE")
print(f"task_id: {sample_example['task_id']}, tool: {sample_example['tool']}, tier: {sample_example['tier']}")
print(f"{'='*70}")

print(f"\n--- PROMPT (snippet) ---\n{prompt_text[:500]}...")

print(f"\n--- ORIGINAL BASE MODEL OUTPUT ---\n{generate_review_merged(original_base_model, merged_tokenizer, prompt_text)}")

print(f"\n--- MERGED SFT MODEL (checkpoint-96) OUTPUT ---\n{generate_review_merged(merged_sft_model, merged_tokenizer, prompt_text)}")

print(f"\n--- REFERENCE (chosen, from your dataset) ---\n{ground_truth_chosen}")

### Final Debugging Step: Saving and Reloading the Fully Merged Model

Given that the previous attempts (loading `PeftModel` directly and using `merge_and_unload` for runtime inference) still resulted in identical outputs, the most conclusive test is to fully save the merged model to disk and then reload it as a standard `AutoModelForCausalLM`. This eliminates any potential runtime issues with LoRA application or `PeftModel` and ensures that the model loaded for inference has the fine-tuned weights permanently baked in.

We will perform the following:
1.  Save the `merged_sft_model` (from the previous step) to a new directory.
2.  Load this saved model and its tokenizer using `AutoModelForCausalLM.from_pretrained` and `AutoTokenizer.from_pretrained`.
3.  Run inference with this fully merged and reloaded model.
4.  Compare its output with the original base model's output and the reference.

In [ ]:
import os

FINAL_MERGED_MODEL_PATH = "/kaggle/working/aicr_sft_merged_model_final"

# Save the merged model (base + LoRA weights) and its tokenizer
merged_sft_model.save_pretrained(FINAL_MERGED_MODEL_PATH)
merged_tokenizer.save_pretrained(FINAL_MERGED_MODEL_PATH)

print(f"Fully merged SFT model saved to: {FINAL_MERGED_MODEL_PATH}")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import json

FINAL_MERGED_MODEL_PATH = "/kaggle/working/aicr_sft_merged_model_final"
MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"

# --- 4-bit quantization config (QLoRA) - Must be the same as during training ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load the original base model for comparison (if not already in memory)
original_base_model_reloaded = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
original_base_model_reloaded.eval()

# Load the tokenizer from the saved merged model path
reloaded_merged_tokenizer = AutoTokenizer.from_pretrained(FINAL_MERGED_MODEL_PATH)
if reloaded_merged_tokenizer.pad_token is None:
    reloaded_merged_tokenizer.pad_token = reloaded_merged_tokenizer.eos_token

# Load the fully merged model (which should now be a standard CausalLM model)
fully_reloaded_sft_model = AutoModelForCausalLM.from_pretrained(
    FINAL_MERGED_MODEL_PATH,
    quantization_config=bnb_config, # Keep quantization for consistency
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
fully_reloaded_sft_model.eval()

print(f"Successfully reloaded original base model and fully merged SFT model from {FINAL_MERGED_MODEL_PATH}")

In [ ]:
# --- Define the generate_review function (re-using for clarity) ---
def generate_review_final(model_to_use, tokenizer_to_use, prompt_text, max_new_tokens=400):
    messages = [{"role": "user", "content": prompt_text}]
    input_text = tokenizer_to_use.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer_to_use(input_text, return_tensors="pt").to(model_to_use.device)

    with torch.no_grad():
        output_ids = model_to_use.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=False,
            pad_token_id=tokenizer_to_use.eos_token_id,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer_to_use.decode(generated, skip_special_tokens=True)

# --- Load a held-out example to test on ---
holdout_examples = []
with open("/kaggle/working/sft_holdout.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        holdout_examples.append(json.loads(line))

# Pick one example for direct comparison (using the same one as before)
sample_example = holdout_examples[0]
prompt_text = sample_example["messages"][0]["content"]
ground_truth_chosen = sample_example["messages"][1]["content"]

print(f"\n{'='*70}")
print(f"FINAL COMPARISON: Original Base Model vs. FULLY RELOADED MERGED SFT Model on HELD-OUT EXAMPLE")
print(f"task_id: {sample_example['task_id']}, tool: {sample_example['tool']}, tier: {sample_example['tier']}")
print(f"{'='*70}")

print(f"\n--- PROMPT (snippet) ---\n{prompt_text[:500]}...")

print(f"\n--- ORIGINAL BASE MODEL OUTPUT (Reloaded) ---\n{generate_review_final(original_base_model_reloaded, reloaded_merged_tokenizer, prompt_text)}")

print(f"\n--- FULLY RELOADED MERGED SFT MODEL OUTPUT ---\n{generate_review_final(fully_reloaded_sft_model, reloaded_merged_tokenizer, prompt_text)}")

print(f"\n--- REFERENCE (chosen, from your dataset) ---\n{ground_truth_chosen}")

In [ ]:
import torch
import json
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"

SYSTEM_MESSAGE = (
    "You are a code reviewer. Follow this exact structure: "
    "1) Briefly validate what works (1-2 sentences), "
    "2) Use 'However' to transition to specific failures, "
    "3) Reference specific test cases (Test 3, Test 5, etc.), "
    "4) Provide concrete fixes. Do NOT provide generic code descriptions or explanations."
)

# --- Load a completely fresh base model, NO LoRA adapter attached ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token

base_only_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
base_only_model.eval()

print("Base model loaded — NO LoRA adapter attached. This is the control.")

# --- Same generation function as before, with system message ---
def generate_review_with_system(model, tokenizer, prompt_text, max_new_tokens=400):
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": prompt_text},
    ]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


# --- Load the same 3 held-out examples used before ---
holdout_examples = []
with open("/kaggle/working/sft_holdout.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        holdout_examples.append(json.loads(line))

target_task_ids = ["task_05", "task_08", "task_23"]
seen = set()
sample_examples = []
for ex in holdout_examples:
    if ex["task_id"] in target_task_ids and ex["task_id"] not in seen:
        sample_examples.append(ex)
        seen.add(ex["task_id"])

# --- Run inference on the base model (control) ---
for i, ex in enumerate(sample_examples):
    prompt_text = ex["messages"][0]["content"]
    ground_truth_chosen = ex["messages"][1]["content"]

    print(f"\n{'='*70}")
    print(f"BASE MODEL (NO LORA) — task_id: {ex['task_id']}, tool: {ex['tool']}, tier: {ex['tier']}")
    print(f"{'='*70}")
    print(f"\n--- BASE MODEL OUTPUT (with system message) ---")
    print(generate_review_with_system(base_only_model, base_tokenizer, prompt_text))
    print(f"\n--- REFERENCE (chosen, from your dataset) ---\n{ground_truth_chosen}")

In [11]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"
SFT_ADAPTER_PATH = "/kaggle/working/aicr-sft-adapter-v2"  

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# --- Load fresh base model ---
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

# --- Attach the LoRA adapter ---
sft_model = PeftModel.from_pretrained(base_model, SFT_ADAPTER_PATH)

# --- DIAGNOSTIC 1: is the adapter actually "active"? ---
print("Active adapters:", sft_model.active_adapters)
print("PEFT config:", sft_model.peft_config)

# --- DIAGNOSTIC 2: are LoRA weights non-zero / non-trivial? ---
for name, param in sft_model.named_parameters():
    if "lora_B" in name and "q_proj" in name:
        print(f"\n{name}")
        print("  shape:", param.shape)
        print("  mean abs value:", param.abs().mean().item())
        print("  max abs value:", param.abs().max().item())
        print("  sample values:", param.flatten()[:5].tolist())
        break

# --- DIAGNOSTIC 3: generate with adapter ACTIVE (normal PeftModel forward) ---
def generate(model, tokenizer, prompt_text, max_new_tokens=200):
    messages = [{"role": "user", "content": prompt_text}]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

test_prompt = "Review this code:\n\ndef add(a, b):\n    return a + b"

print("\n--- Generation WITH adapter active ---")
sft_model.eval()
output_with_adapter = generate(sft_model, tokenizer, test_prompt)
print(output_with_adapter)

# --- DIAGNOSTIC 4: explicitly DISABLE the adapter and regenerate ---
with sft_model.disable_adapter():
    print("\n--- Generation WITH adapter EXPLICITLY DISABLED (should look like base model) ---")
    output_without_adapter = generate(sft_model, tokenizer, test_prompt)
    print(output_without_adapter)

print("\n--- Are the two outputs identical? ---")
print(output_with_adapter == output_without_adapter)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Active adapters: ['default']
PEFT config: {'default': LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.19.1', base_model_name_or_path='Qwen/Qwen2.5-Coder-3B-Instruct', revision=None, inference_mode=True, r=16, target_modules={'o_proj', 'v_proj', 'gate_proj', 'down_proj', 'up_proj', 'k_proj', 'q_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=Non